# 08 — Model Evaluation and Error Analysis

Final consolidation. This notebook only *reads* artifacts produced by notebooks 04–07 and reloads the ResNet50 for image-level error analysis. No model is retrained and no metric is typed in by hand — every number comes from a saved table or a recomputed prediction.

Three headline tables (classification, regression, incremental value), then error analysis on both tasks with attention to the heavy tail of claim severity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

TABLE_DIR = Path("../outputs/tables")
PRED_DIR = Path("../outputs/predictions")
MODEL_DIR = Path("../models")

DATA_DIR = Path("c:\\Users\\Niraj Mhatre\\projects\\motor_insurance_data\\Fast_Furious_Insured")
TRAIN_IMG_DIR = DATA_DIR / "trainImages"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 8.1 Classification comparison

In [ ]:
def safe_read(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print("Missing:", path)
        return None

baseline_cls = safe_read(TABLE_DIR / "baseline_cnn_val_metrics.csv")
resnet_cls = safe_read(TABLE_DIR / "resnet50_val_metrics.csv")

classification_table = pd.concat([baseline_cls, resnet_cls], ignore_index=True)
classification_table = classification_table.set_index("model").round(4)
classification_table

## 8.2 Regression comparison (tabular only)

In [ ]:
regression_table = safe_read(TABLE_DIR / "regression_tabular_metrics.csv").set_index("model").round(2)
regression_table

## 8.3 Incremental value of computer vision

In [ ]:
incremental_table = safe_read(TABLE_DIR / "incremental_value_metrics.csv").set_index("Feature Set").round(2)
incremental_table

## 8.4 Classification error analysis

Reload the ResNet50 and the saved validation probabilities. We surface the most confident mistakes — false positives (intact vehicle flagged as damaged) and false negatives (damage missed) — and view the images to reason about *why* the model erred.

In [ ]:
resnet_preds = safe_read(PRED_DIR / "resnet50_val_preds.csv")

resnet_preds["pred"] = (resnet_preds["cnn_damage_prob"] >= 0.5).astype(int)

false_pos = resnet_preds[(resnet_preds["Condition"] == 0) & (resnet_preds["pred"] == 1)]
false_neg = resnet_preds[(resnet_preds["Condition"] == 1) & (resnet_preds["pred"] == 0)]

false_pos = false_pos.sort_values("cnn_damage_prob", ascending=False)
false_neg = false_neg.sort_values("cnn_damage_prob", ascending=True)

print("False positives:", len(false_pos), "| False negatives:", len(false_neg))

In [ ]:
def show_error_grid(frame, title, n=4):
    frame = frame.head(n)
    if len(frame) == 0:
        print("None to show for:", title)
        return
    fig, axes = plt.subplots(1, len(frame), figsize=(4 * len(frame), 4))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, frame.iterrows()):
        img = Image.open(TRAIN_IMG_DIR / row["Image_path"]).convert("RGB")
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"actual {int(row['Condition'])} | P(dmg) {row['cnn_damage_prob']:.2f}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_error_grid(false_pos, "Most confident false positives")
show_error_grid(false_neg, "Most confident false negatives")

## 8.5 Regression error analysis

From the combined test predictions, the largest absolute and percentage errors. Percentage error is reported because a fixed rupee error means something very different on a small claim than on a large one.

In [ ]:
combined = safe_read(PRED_DIR / "combined_test_preds.csv")

combined["abs_err"] = (combined["Amount"] - combined["pred_tabular_plus_cnn"]).abs()
combined["pct_err"] = combined["abs_err"] / combined["Amount"].replace(0, np.nan) * 100

print("Largest absolute errors:")
display_cols = ["Image_path", "Condition", "cnn_damage_prob", "Amount",
                "pred_tabular", "pred_tabular_plus_cnn", "abs_err", "pct_err"]
combined.sort_values("abs_err", ascending=False)[display_cols].head(10)

In [ ]:
print("Largest percentage errors (claims above the median):")
median_amt = combined["Amount"].median()
combined[combined["Amount"] > median_amt].sort_values("pct_err", ascending=False)[display_cols].head(10)

## 8.6 Performance across the loss distribution

Whether the model under-prices large claims is the question that matters for reserving adequacy. We bucket the test claims into small / medium / large and compare MAE for the tabular and enhanced models side by side.

In [ ]:
valid = combined.dropna(subset=["Amount"]).copy()
valid["bucket"] = pd.qcut(valid["Amount"], q=[0, 0.5, 0.9, 1.0],
                          labels=["small", "medium", "large"], duplicates="drop")

grouped = valid.groupby("bucket", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "MAE_tabular": (g["Amount"] - g["pred_tabular"]).abs().mean(),
        "MAE_tabular_plus_cnn": (g["Amount"] - g["pred_tabular_plus_cnn"]).abs().mean(),
    })
)
grouped

In [ ]:
ax = grouped[["MAE_tabular", "MAE_tabular_plus_cnn"]].plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("Mean absolute error")
ax.set_title("Error by claim-size bucket: tabular vs tabular + CNN")
plt.tight_layout()
plt.show()

## 8.7 Summary

The three tables (8.1–8.3) are the project's quantitative result. The error analysis explains *where* the models succeed and fail: which images fool the classifier, which claims the regressor misprices, and whether the vision feature helps most exactly where tabular models are weakest — the large-claim tail. Conclusions are stated strictly from these held-out numbers.